# Ingesta de datos desde mysql

In [1]:
!pip install notebook findspark


[notice] A new release of pip is available: 23.0.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!python --version

Python 3.10.20


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder\
    .master("spark://spark-master:7077")\
    .appName("Ingesta")\
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")\
    .getOrCreate()    

26/08/04 03:40:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
import pyspark
import os

print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)

PySpark: 3.5.1
Spark: 3.5.1


In [6]:
!ls /opt/spark/jars/mysql-connector-j-8.4.0.jar

/opt/spark/jars/mysql-connector-j-8.4.0.jar


In [8]:
df = (spark.read
         .format("jdbc")
         .option("url", "jdbc:mysql://mysql:3306/retail_db")
         .option("dbtable", "categories")
         .option("user", "root")
         .option("password", "root")
         .option("driver", "com.mysql.cj.jdbc.Driver")
         .load())
df.show()

+-----------+----------------------+-------------------+
|category_id|category_department_id|      category_name|
+-----------+----------------------+-------------------+
|          1|                     2|           Football|
|          2|                     2|             Soccer|
|          3|                     2|Baseball & Softball|
|          4|                     2|         Basketball|
|          5|                     2|           Lacrosse|
|          6|                     2|   Tennis & Racquet|
|          7|                     2|             Hockey|
|          8|                     2|        More Sports|
|          9|                     3|   Cardio Equipment|
|         10|                     3|  Strength Training|
|         11|                     3|Fitness Accessories|
|         12|                     3|       Boxing & MMA|
|         13|                     3|        Electronics|
|         14|                     3|     Yoga & Pilates|
|         15|                  

# Reto de clase
#### 1 realizar un procesamiento de join

#### 2 realizar iun procesamiento de agrupacion

#### 3 escribirlo a una capa cleansed 

#### 4 escribir sobre la misma base retail_db con el nombre de tabla cleansed_agregado

In [9]:
df1 = (spark.read
         .format("jdbc")
         .option("url", "jdbc:mysql://mysql:3306/retail_db")
         .option("dbtable", "products")
         .option("user", "root")
         .option("password", "root")
         .option("driver", "com.mysql.cj.jdbc.Driver")
         .load())
df1.show()

+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|product_id|product_category_id|        product_name|product_description|product_price|       product_image|
+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|         1|                  2|Quest Q64 10 FT. ...|                   |        59.98|http://images.acm...|
|         2|                  2|Under Armour Men'...|                   |       129.99|http://images.acm...|
|         3|                  2|Under Armour Men'...|                   |        89.99|http://images.acm...|
|         4|                  2|Under Armour Men'...|                   |        89.99|http://images.acm...|
|         5|                  2|Riddell Youth Rev...|                   |       199.99|http://images.acm...|
|         6|                  2|Jordan Men's VI R...|                   |       134.99|http://images.acm...|
|         7|       

In [10]:
df.createOrReplaceTempView("categorias")
df1.createOrReplaceTempView("productos")

In [11]:
df2 = spark.sql("""
    SELECT
        c.category_id,
        c.category_name,
        p.product_id,
        p.product_name,
        p.product_category_id
    FROM categorias c
    INNER JOIN productos p
    ON c.category_id = p.product_category_id
""")

df2.printSchema()
df2.show()

root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category_id: integer (nullable = true)



+-----------+------------------+----------+--------------------+-------------------+
|category_id|     category_name|product_id|        product_name|product_category_id|
+-----------+------------------+----------+--------------------+-------------------+
|         31|Women's Golf Clubs|       669|Cleveland Golf My...|                 31|
|         31|Women's Golf Clubs|       670|Cleveland Golf El...|                 31|
|         31|Women's Golf Clubs|       671|Cleveland Golf Co...|                 31|
|         31|Women's Golf Clubs|       672|     PING G30 Driver|                 31|
|         31|Women's Golf Clubs|       673|PING G30 Fairway ...|                 31|
|         31|Women's Golf Clubs|       674|     PING G30 Hybrid|                 31|
|         31|Women's Golf Clubs|       675|PING G30 Irons - ...|                 31|
|         31|Women's Golf Clubs|       676|PING G30 Irons - ...|                 31|
|         31|Women's Golf Clubs|       677|TaylorMade White ...| 

In [12]:
df1 = (spark.read
       .format("jdbc")
       .option("url", "jdbc:mysql://mysql:3306/retail_db")
       .option("dbtable", "products")
       .option("user", "root")
       .option("password", "root")
       .option("driver", "com.mysql.cj.jdbc.Driver")
       .option("partitionColumn", "product_id")
       .option("lowerBound", "1")
       .option("upperBound", "100000")
       .option("numPartitions", "4")
       .load())

In [13]:
df1.rdd.getNumPartitions()

4

In [14]:
print("Particiones:", df1.rdd.getNumPartitions())
print("Paralelismo:", spark.sparkContext.defaultParallelism)

Particiones: 4
Paralelismo: 2


In [15]:
from pyspark.sql.functions import sum

df1.groupBy("product_category_id") \
   .agg(sum("product_id").alias("total_product_id")) \
   .show()

[Stage 8:=============================>                             (2 + 2) / 4]

+-------------------+----------------+
|product_category_id|total_product_id|
+-------------------+----------------+
|                 31|           16332|
|                 53|           28548|
|                 34|           18060|
|                 26|           13452|
|                 27|           14028|
|                 44|           23364|
|                 12|            6060|
|                 22|           11724|
|                 47|           25092|
|                 52|           27972|
|                 13|            6636|
|                  6|            2604|
|                 16|            8364|
|                  3|             876|
|                 20|           10572|
|                 40|           21516|
|                 57|           30852|
|                 54|           29124|
|                 48|           25668|
|                  5|            2028|
+-------------------+----------------+
only showing top 20 rows



In [16]:
df_final=df.join(df1,df.category_id==df1.product_category_id,'inner')

In [17]:
df_final.printSchema()

root
 |-- category_id: integer (nullable = true)
 |-- category_department_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_category_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- product_price: double (nullable = true)
 |-- product_image: string (nullable = true)



In [18]:
df_final.createOrReplaceTempView("PERSON_DATA")

In [19]:
df2 = spark.sql("SELECT * from PERSON_DATA")
df2.printSchema()
df2.show()

root
 |-- category_id: integer (nullable = true)
 |-- category_department_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_category_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- product_price: double (nullable = true)
 |-- product_image: string (nullable = true)



+-----------+----------------------+------------------+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|category_id|category_department_id|     category_name|product_id|product_category_id|        product_name|product_description|product_price|       product_image|
+-----------+----------------------+------------------+----------+-------------------+--------------------+-------------------+-------------+--------------------+
|         31|                     6|Women's Golf Clubs|       669|                 31|Cleveland Golf My...|                   |       179.99|http://images.acm...|
|         31|                     6|Women's Golf Clubs|       670|                 31|Cleveland Golf El...|                   |       209.99|http://images.acm...|
|         31|                     6|Women's Golf Clubs|       671|                 31|Cleveland Golf Co...|                   |       209.99|http://images.acm...|
|         31|         

In [20]:
df2.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'UnresolvedRelation [PERSON_DATA], [], false

== Analyzed Logical Plan ==
category_id: int, category_department_id: int, category_name: string, product_id: int, product_category_id: int, product_name: string, product_description: string, product_price: double, product_image: string
Project [category_id#0, category_department_id#1, category_name#2, product_id#82, product_category_id#83, product_name#84, product_description#85, product_price#86, product_image#87]
+- SubqueryAlias person_data
   +- View (`PERSON_DATA`, [category_id#0,category_department_id#1,category_name#2,product_id#82,product_category_id#83,product_name#84,product_description#85,product_price#86,product_image#87])
      +- Join Inner, (category_id#0 = product_category_id#83)
         :- Relation [category_id#0,category_department_id#1,category_name#2] JDBCRelation(categories) [numPartitions=1]
         +- Relation [product_id#82,product_category_id#83,product_name#84,product_de

In [21]:
df2.write.mode("overwrite").csv(
    "hdfs://namenode:8020/datalake/raw/categories_agregado"
)

In [22]:
df2.write \
    .format("jdbc") \
    .mode("overwrite") \
    .option("url", "jdbc:mysql://mysql:3306/retail_db") \
    .option("dbtable", "categories_agregado") \
    .option("user", "root") \
    .option("password", "root") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .save()

In [23]:
print("Particiones:", df2.rdd.getNumPartitions())
print("Paralelismo Spark:", spark.sparkContext.defaultParallelism)

Particiones: 1
Paralelismo Spark: 2


In [24]:
df2_parallel=df2.repartition(8)

In [25]:
print("Particiones:", df2_parallel.rdd.getNumPartitions())
print("Paralelismo Spark:", spark.sparkContext.defaultParallelism)

Particiones: 8
Paralelismo Spark: 2


In [26]:
(df2_parallel.write
    .format("jdbc")
    .mode("overwrite")
    .option("url", "jdbc:mysql://mysql:3306/retail_db")
    .option("dbtable", "categories_agregado")
    .option("user", "root")
    .option("password", "root")
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .option("batchsize", "1000")
    .save())

In [27]:
df2.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("hdfs://namenode:8020/datalake/raw/categories_agregado")

In [28]:
!java -version

openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-deb12u1-Debian)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-deb12u1-Debian, mixed mode, sharing)


In [28]:
df = spark.read \
   .option("header", "true") \
   .option("inferSchema", "true") \
   .csv("hdfs://namenode:8020/datalake/raw/categories_agregado")

In [29]:
df.count()

1321

In [30]:
spark.sparkContext.uiWebUrl

'http://jupyter:4040'

In [35]:
print(spark.sparkContext.uiWebUrl)

http://7349b16553d6:4041
